# 에이전틱 쿼리 생성 + 정적 검증 — 실측 프로토타입

> 실행 2026-08-16 · `gpt-4o-mini` · 설계 캐논: `.personal/strategy/RAG_05_에이전틱_쿼리_생성_설계.md`(이하 RAG_05)
> 선행: `retrieval_strategy_evaluation.ipynb`(이하 RAG_04 평가 노트북) — 이 노트북은 그 노트북이 이미
> 만들어 둔 `STAGE2_SCENARIOS`(8건, 실제 커밋된 라벨)를 그대로 재사용해 **네 번째 arm**을 추가한다.

## 이 노트북이 답하려는 질문

| # | 질문 |
|---|---|
| ① | 조립기(파이썬 템플릿) 대신 LLM이 검색 질의문을 직접 쓰게 하면 검색 품질이 나아지는가 |
| ② | 정적 검증기가 과거 실제 프로덕션 버그(ML 피처명 누출 등)를 실제로 잡아내는가 |
| ③ | 검증 실패 → 재시도 → 폴백 경로가 실제로 필요한 만큼 발동하는가 |

## ⚠️ 먼저 밝혀 두는 한계

1. **표본 8건, 합성 시나리오** — RAG_04 평가 노트북이 만든 것과 동일한 셋을 그대로 재사용했다(새
   규정 사실을 지어내지 않는다 — `tiger_inc/` 원문을 이 노트북에서 열지 않았다).
2. **재시도/폴백 경로 미검증** — 이번 실행에서 8건 전부 1차 시도에 검증을 통과했다. 즉 "검증기가
   통과시키는 정상 케이스"만 실측됐고, "거절당한 뒤 LLM이 실제로 잘 고치는지"는 이 노트북이 답하지
   못한다(RAG_05 §3-②).
3. **재현성** — LLM 호출 셀은 `temperature=0.2`라도 완전한 결정론은 아니다. 아래 출력은 최초 실행
   결과를 그대로 보존한 것이고, 재실행하면 생성되는 문구는 달라질 수 있다(질적 경향은 재현될 것으로
   기대하지만 확인되지 않았다). 재실행 시 OpenAI 과금이 발생한다(8건 기준 소액, gpt-4o-mini).

---
## 0. 환경 설정

컨테이너 밖(호스트 파이썬)에서 돌릴 때만 `CHROMA_HOST` 오버라이드가 필요하다 — 컨테이너 안에서
돌린다면 `app/config.py`의 기본값(`chroma`:8000)이 이미 맞다. 이 노트북은
`docker compose exec ai jupyter ...` 또는 동등한 컨테이너 내부 실행을 전제로 작성됐다.

In [1]:
import re
import time

import numpy as np
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel

from app.config import settings
from app.ml.features import FEATURE_COLUMNS
from app.rag.embedding import store

MODEL = "gpt-4o-mini"
client = OpenAI(api_key=settings.openai_api_key)
CLIENT = store.get_client()

print(f"OPENAI_API_KEY   {'설정됨' if settings.openai_api_key else '⚠️ 없음'}")
print(f"CHROMA_HOST      {settings.chroma_host}:{settings.chroma_port}")
print(f"FEATURE_COLUMNS  {len(FEATURE_COLUMNS)}개 (검증기 블랙리스트 원천 — app/ml/features.py)")

OPENAI_API_KEY   설정됨
CHROMA_HOST      chroma:8000
FEATURE_COLUMNS  15개 (검증기 블랙리스트 원천 — app/ml/features.py)


---
## 1. 시나리오 — `retrieval_strategy_evaluation.ipynb`의 `STAGE2_SCENARIOS` 재사용

`(scenario_id, category, merchant, feature_hint_raw, feature_hint_nl, facts_nl, relevant_units)`.
`feature_hint_raw`는 지금 production `_build_query()`가 실제로 쓰는 blob 재료(ML 피처 이름 그대로),
`feature_hint_nl`은 그걸 사람 말로 풀어 쓴 버전, `facts_nl`은 그 조문과 겹치는 판정 사실만 채웠다
(모르는 건 비워 둔다 — 알고 있는 척 지어내지 않는다).

In [1]:
# (scenario_id, category, merchant, feature_hint_raw, feature_hint_nl, facts_nl, relevant_units)
# 원본: retrieval_strategy_evaluation.ipynb §8 STAGE2_SCENARIOS — 그대로 재사용(새 규정 사실 없음)
STAGE2_SCENARIOS = [
    ("S01", "회식", "강남모던바", "최근7일사용횟수",
     "최근 일주일간 결제 횟수가 급증한 점", "참석 인원 2명",
     ["회식_사용규정|제6조"]),
    ("S02", "업무추진비", "한정식당 청담점", "거래금액_Zscore_확장",
     "평소 대비 결제 금액이 크게 높은 점", "청탁금지법 대상자 참석, 지출유형 식사",
     ["업무추진비_사용규정|제9조"]),
    ("S03", "출장비", "OO헤어살롱", "승인시간대",
     "통상적이지 않은 업종·시간대 결제라는 점", "",
     ["출장비_사용규정|제11조"]),
    ("S04", "업무추진비", "고급레스토랑", "카드누적사용액",
     "카드 누적 사용액이 이례적으로 큰 점", "",
     ["법인카드_사용규정|제14조"]),
    ("S05", "업무추진비", "일식오마카세", "최근7일사용횟수",
     "동일 거래처 반복 결제 빈도가 높은 점", "",
     ["업무추진비_사용규정|제13조"]),
    ("S06", "업무추진비", "명품매장", "통합승인금액",
     "결제 금액이 이례적으로 큰 점", "지출유형 선물, 청탁금지법 대상자 참석",
     ["업무추진비_사용규정|별표1"]),
    ("S07", "출장비", "OO호텔", "거래금액_Zscore_확장",
     "평소 대비 숙박 결제 금액이 높은 점", "지출유형 숙박",
     ["출장비_사용규정|별표1"]),
    ("S08", "회식", "이자카야", "카드첫거래여부",
     "해당 가맹점과의 첫 거래라는 점", "사전승인 미확인",
     ["회식_사용규정|제4조", "회식_사용규정|별표1"]),
]
print(f"{len(STAGE2_SCENARIOS)}개 시나리오 로드")

8개 시나리오 로드


---
## 2. 정적 검증기 — RAG_05 §2.3

**블랙리스트**다(RAG_04 §2.3의 화이트리스트 "모르는 유형은 멈춘다"와 반대 극성). 이유는 소비 주체가
다르기 때문 — 파이썬 조립기는 스스로 모르는 값을 만들 수 없어 화이트리스트가 자연스러웠지만, LLM은
자유 문장을 쓰므로 "정상 한국어 문장"을 전부 열거할 수 없다. 대신 **알려진 실패 모드만** 걸러낸다.

In [1]:
CODE_SENTINELS = {"GLOBAL", "TEST_DEMO"}
FEATURE_TOKEN_RE = re.compile("|".join(re.escape(c) for c in FEATURE_COLUMNS))
META_PREAMBLE_RE = re.compile(r'^\s*["\'`]|검색어\s*[:：]|^\s*(다음은|네,|물론입니다)')
CODE_FENCE_RE = re.compile(r"```")


def validate_query(q: str) -> tuple[bool, str | None]:
    """규칙 순서 = 저렴한 검사 먼저. 첫 번째 위반 사유만 반환(중첩 위반은 재시도 때 순차 노출)."""
    if not q or not q.strip():
        return False, "EMPTY_QUERY"
    if len(q.strip()) < 4:
        return False, "TOO_SHORT"
    if len(q) > 200:
        return False, "TOO_LONG"
    if "\n" in q or CODE_FENCE_RE.search(q):
        return False, "MULTILINE_OR_CODE_FENCE"
    if META_PREAMBLE_RE.search(q):
        return False, "META_PREAMBLE"
    for sentinel in CODE_SENTINELS:
        if re.search(rf"\b{sentinel}\b", q):
            return False, f"CODE_SENTINEL_LEAK:{sentinel}"
    m = FEATURE_TOKEN_RE.search(q)
    if m:
        return False, f"RAW_FEATURE_NAME_LEAK:{m.group(0)}"
    return True, None

### 2.1 단위 테스트 — 과거 실제 사고를 회귀로 잡는가

RAG_04 §1.2가 지적한 실제 프로덕션 버그(현재 `_build_query()`가 내는 blob 쿼리)를 그대로 넣어서,
검증기가 잡아내는지 확인한다. API 비용 없음(순수 로직).

In [1]:
VALIDATOR_UNIT_TESTS = [
    ("빈 문자열", "", False, "EMPTY_QUERY"),
    ("공백만", "   ", False, "EMPTY_QUERY"),
    ("기존 production blob (S01)", "회식 강남모던바 최근7일사용횟수",
     False, "RAW_FEATURE_NAME_LEAK:최근7일사용횟수"),
    ("기존 production blob (S02)", "업무추진비 한정식당 청담점 거래금액_Zscore_확장",
     False, "RAW_FEATURE_NAME_LEAK:거래금액_Zscore_확장"),
    ("scope sentinel 누출", "GLOBAL 복리후생비 회의비 구분 기준",
     False, "CODE_SENTINEL_LEAK:GLOBAL"),
    ("LLM 메타발화", "검색어: 회식비 참석 인원 2명 인정 기준",
     False, "META_PREAMBLE"),
    ("정상 자연어 쿼리",
     "강남모던바에서 발생한 회식비 결제인데 참석 인원이 2명입니다. "
     "회식비로 인정되는 최소 인원 기준을 확인해야 합니다.",
     True, None),
]


def run_validator_unit_tests() -> pd.DataFrame:
    rows = []
    for name, q, expect_ok, expect_reason in VALIDATOR_UNIT_TESTS:
        ok, reason = validate_query(q)
        passed = (ok == expect_ok) and (reason == expect_reason or (ok and expect_ok))
        rows.append({"case": name, "query": q, "validator_ok": ok, "reason": reason, "test_passed": passed})
    return pd.DataFrame(rows)


VT = run_validator_unit_tests()
display(VT)
print(f"\n전부 통과: {VT.test_passed.all()}  ({VT.test_passed.sum()}/{len(VT)})")

전부 통과: True  (7/7)


**7/7 통과.** 지금 프로덕션이 실제로 내는 그 blob 쿼리 2건(S01·S02 재료)이 검증기를 통과하지
못한다 — 이 검증기를 파이프라인에 얹기만 해도 RAG_04 §1.2의 그 버그는 최소한 조용히 넘어가지
않는다.

---
## 3. 에이전틱 쿼리 생성 — RAG_05 §2.1

LLM에게 주는 가이드(시스템 프롬프트)는 §2의 검증 규칙과 **1:1로 대응**한다 — 규칙 2·3이 곧
`RAW_FEATURE_NAME_LEAK`/`CODE_SENTINEL_LEAK`이다. 검증기가 거절하면 그 사유를 그대로 LLM에게
피드백으로 돌려주므로(아래 `generate_query_agentic`의 `feedback` 인자), 가이드 문장과 검증기
사유 문자열이 어긋나 있으면 재시도가 무의미해진다.

In [1]:
QUERY_GUIDE_SYSTEM_PROMPT = """당신은 법인카드 정산 Risk Review 2차 검증 단계에서, 사내 규정 벡터
검색(policy_docs)에 던질 검색 질의문을 작성하는 역할입니다.

반드시 지켜야 할 규칙:
1. 실제 담당자가 규정을 찾을 때 쓰는 자연스러운 한국어 문장으로 작성하세요. 키워드를 공백으로
   나열하지 마세요(예: "회식 강남모던바 최근7일사용횟수" ← 이런 형태 금지).
2. ML 이상탐지 모델의 내부 피처 이름(예: 최근7일사용횟수, 거래금액_Zscore_확장, 카드누적사용액 같은
   변수명)을 문장에 그대로 쓰지 마세요 — 규정 문서에 없는 어휘라 검색에 오히려 방해가 됩니다.
   대신 그 의미를 사람 말로 풀어서 쓰세요(예: "최근 결제 횟수가 급증했다").
3. 시스템 내부 코드(GLOBAL 같은 scope 상수)를 문장에 그대로 쓰지 마세요.
4. 알고 있는 사실(거래 사실)이 있으면 문장에 자연스럽게 녹이세요. 모르는 사실은 지어내지 말고
   생략하세요.
5. 출력은 검색 질의문 한 줄만 반환하세요. "검색어:" 같은 접두사, 설명, 따옴표, 코드블록을
   붙이지 마세요.

예시 (좋음): "강남모던바에서 발생한 회식비 결제인데 참석 인원이 2명입니다. 회식비로 인정되는
최소 인원 기준을 확인하고 싶습니다."
예시 (나쁨): "회식 강남모던바 최근7일사용횟수" ← 키워드 나열, 피처명 누출로 금지"""


class GeneratedQuery(BaseModel):
    query: str


def generate_query_agentic(category, merchant, feature_hint_nl, facts_nl, *, feedback=None):
    user_prompt = (
        f"[검토 대상] 가맹점: {merchant} / 분류: {category}\n"
        f"[이상탐지가 지목한 사유] {feature_hint_nl}\n"
        f"[알고 있는 거래 사실] {facts_nl or '(없음)'}\n"
        "위 정보를 바탕으로, 이 건이 규정 위반인지 확인하기 위한 검색 질의문을 작성하세요."
    )
    if feedback:
        user_prompt += f"\n\n[이전 시도 실패 사유] {feedback} — 이 문제를 고쳐서 다시 작성하세요."
    resp = client.beta.chat.completions.parse(
        model=MODEL, temperature=0.2, timeout=30,
        messages=[
            {"role": "system", "content": QUERY_GUIDE_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        response_format=GeneratedQuery,
    )
    parsed = resp.choices[0].message.parsed
    return parsed.query if parsed else ""


def build_query_facts_fallback(category, merchant, feature_hint_nl, facts_nl):
    """검증 실패 시 폴백 — RAG_04 §2.1 채택안(자연어+facts)과 동일 템플릿."""
    base = f"{merchant}에서 발생한 {category} 결제가 이상거래로 탐지됐습니다. {feature_hint_nl}."
    if facts_nl:
        base += f" 거래 사실: {facts_nl}."
    base += " 규정 위반 여부를 확인해야 합니다."
    return base

### 3.1 채점 함수 — RAG_04 평가 노트북 `score_stage2`와 동일 로직

같은 채점 함수를 그대로 재사용해야 이번 arm과 기존 3개 arm(blob/자연어/자연어+facts)이
apples-to-apples 비교가 된다.

In [1]:
def unit_of(metadata: dict) -> str:
    doc, art = metadata.get("doc_name"), metadata.get("article_label")
    if doc and art:
        return f"{doc}|{art}"
    return metadata.get("citation") or "(unit 미상)"


def score_stage2(query: str, relevant: list[str], *, top_k: int = 10) -> dict:
    hits = store.search(query, collection_name="policy_docs", top_k=top_k, client=CLIENT, expand_parent=False)
    relset = set(relevant)
    ranked, seen = [], set()
    for h in hits:
        u = unit_of(h["metadata"])
        if u not in seen:
            seen.add(u)
            ranked.append(u)
    row = {}
    for k in (1, 3, 5, 10):
        hit = relset & set(ranked[:k])
        row[f"recall@{k}"] = len(hit) / len(relset)
    rr = next((1 / (i + 1) for i, u in enumerate(ranked) if u in relset), 0.0)
    row["mrr"] = rr
    dcg = sum(1 / np.log2(i + 2) for i, u in enumerate(ranked[:10]) if u in relset)
    idcg = sum(1 / np.log2(i + 2) for i in range(min(len(relset), 10)))
    row["ndcg@10"] = dcg / idcg if idcg else 0.0
    row["top1_unit"] = ranked[0] if ranked else "(결과 없음)"
    return row


def run_agentic_arm(max_retries: int = 1) -> pd.DataFrame:
    """generate -> validate -> (실패 시 피드백과 함께 1회 재시도) -> (그래도 실패면 폴백) -> retrieve."""
    rows = []
    for sid, cat, merchant, feat_raw, feat_nl, facts_nl, rel in STAGE2_SCENARIOS:
        feedback = None
        query, ok, reason, attempts, fallback_used = "", False, None, 0, False
        for attempt in range(max_retries + 1):
            attempts = attempt + 1
            query = generate_query_agentic(cat, merchant, feat_nl, facts_nl, feedback=feedback)
            ok, reason = validate_query(query)
            if ok:
                break
            feedback = reason
        if not ok:
            query = build_query_facts_fallback(cat, merchant, feat_nl, facts_nl)
            fallback_used = True
        score = score_stage2(query, rel)
        score.update(scenario_id=sid, query=query, validator_ok=ok, validator_reason=reason,
                     attempts=attempts, fallback_used=fallback_used)
        rows.append(score)
    return pd.DataFrame(rows)

### 3.2 실행 — 실제 LLM 호출 8건

⚠️ 아래 셀은 OpenAI API를 실제로 호출한다(gpt-4o-mini, 8건 기준 소액 과금, 이번 실행 약 11초).
출력은 2026-08-16 최초 실행 결과를 그대로 보존했다 — 재실행하면 문구는 달라질 수 있다.

In [1]:
t0 = time.time()
AGENTIC_DF = run_agentic_arm()
print(f"완료 — {time.time() - t0:.1f}s\n")
display(AGENTIC_DF[["scenario_id", "mrr", "recall@1", "ndcg@10", "top1_unit",
                     "validator_ok", "attempts", "fallback_used"]])

print("\n--- 생성된 쿼리 전문 ---")
for _, r in AGENTIC_DF.iterrows():
    print(f"[{r.scenario_id}] (fallback={r.fallback_used}) {r.query}")

완료 — 10.9s

--- 생성된 쿼리 전문 ---
[S01] (fallback=False) 강남모던바에서의 회식비 결제가 최근 일주일 동안 빈번하게 발생했습니다. 참석 인원이 2명인 경우 회식비로 인정되는 기준을 확인하고 싶습니다.
[S02] (fallback=False) 한정식당 청담점에서의 업무추진비 지출이 평소보다 많이 발생했는데, 청탁금지법 대상자가 참석한 식사 비용이 규정에 어긋나는지 확인하고 싶습니다.
[S03] (fallback=False) OO헤어살롱에서 발생한 출장비 결제인데, 결제 시간대가 일반적인 출장비 사용 시간과 다릅니다. 이 경우 규정 위반 여부를 확인하고 싶습니다.
[S04] (fallback=False) 고급레스토랑에서 발생한 업무추진비 결제인데, 카드 사용액이 비정상적으로 높습니다. 이 경우 규정 위반 여부를 확인하고 싶습니다.
[S05] (fallback=False) 일식오마카세에서 반복적으로 결제한 경우 업무추진비로 인정될 수 있는지 규정을 확인하고 싶습니다.
[S06] (fallback=False) 명품매장에서 발생한 업무추진비 결제 금액이 이례적으로 큰데, 이 지출이 청탁금지법에 저촉되는지 확인해야 합니다.
[S07] (fallback=False) OO호텔에서 발생한 숙박비 결제 금액이 평소보다 높습니다. 출장비로 인정되는 숙박비의 기준을 확인해야 합니다.
[S08] (fallback=False) 이자카야에서의 회식비 결제는 사전 승인이 필요하다고 알고 있습니다. 첫 거래인 경우에도 사전 승인이 반드시 필요한지 확인하고 싶습니다.


**8/8 시나리오가 1차 시도에서 검증을 통과했다**(재시도·폴백 0건). LLM이 가이드를 잘 따랐다는
뜻이지만, RAG_05 §3-②가 짚은 대로 **재시도 경로 자체는 이번 실행으로 검증되지 않았다** — 검증기가
정상 케이스를 통과시키는 것만 확인됐다.

---
## 4. 기존 3개 arm과 비교

`retrieval_strategy_evaluation.ipynb`가 같은 8개 시나리오·같은 `score_stage2`로 이미 실행해 저장한
`docling_eval/output/retrieval/stage2_{blob,natural,facts}.csv`를 그대로 불러온다(재실행 안 함 —
같은 코드·같은 시나리오라 다시 돌려도 값이 바뀌지 않는 결정론적 arm들이다, LLM이 개입하는 건
`facts`/`natural`이 아니라 이번 `agentic` arm뿐).

In [1]:
import pathlib
STAGE2_DIR = pathlib.Path("/data/docling_eval/output/retrieval")

baseline = {name: pd.read_csv(STAGE2_DIR / f"stage2_{name}.csv").sort_values("scenario_id").reset_index(drop=True)
            for name in ("blob", "natural", "facts")}
agentic_sorted = AGENTIC_DF.sort_values("scenario_id").reset_index(drop=True)

SUMMARY = pd.DataFrame([
    {"방식": "블롭(현재 production)", "MRR": baseline["blob"].mrr.mean(),
     "Recall@1": baseline["blob"]["recall@1"].mean(), "nDCG@10": baseline["blob"]["ndcg@10"].mean()},
    {"방식": "자연어 문장화(조립기)", "MRR": baseline["natural"].mrr.mean(),
     "Recall@1": baseline["natural"]["recall@1"].mean(), "nDCG@10": baseline["natural"]["ndcg@10"].mean()},
    {"방식": "자연어+facts(조립기, RAG_04 채택안)", "MRR": baseline["facts"].mrr.mean(),
     "Recall@1": baseline["facts"]["recall@1"].mean(), "nDCG@10": baseline["facts"]["ndcg@10"].mean()},
    {"방식": "에이전틱 생성+정적검증(이번 안)", "MRR": agentic_sorted.mrr.mean(),
     "Recall@1": agentic_sorted["recall@1"].mean(), "nDCG@10": agentic_sorted["ndcg@10"].mean()},
]).round(4)
display(SUMMARY)

                              방식     MRR  Recall@1  nDCG@10
0             블롭(현재 production)  0.1917     0.125   0.2359
1                  자연어 문장화(조립기)  0.2929     0.250   0.3400
2       자연어+facts(조립기, RAG_04 채택안)  0.3899     0.250   0.4867
3         에이전틱 생성+정적검증(이번 안)  0.4792     0.250   0.5571


### 4.1 통계적으로 유의한가 — RAG_04 채택안(facts) 대비 짝비교 부트스트랩 CI

점추정 최고는 이번 arm이지만, RAG_04 §1.2a가 채택 기준으로 삼은 것과 같은 잣대(짝비교 부트스트랩
95% CI가 0을 미포함해야 "유의")를 그대로 적용한다.

In [1]:
SEED = 42
delta = agentic_sorted.mrr.values - baseline["facts"].mrr.values
print("per-scenario Δ(agentic-facts):", np.round(delta, 3).tolist())
print(f"평균 ΔMRR = {delta.mean():+.4f}")

rng = np.random.default_rng(SEED)
idx = rng.integers(0, len(delta), size=(5000, len(delta)))
means = delta[idx].mean(axis=1)
lo, hi = np.quantile(means, 0.025), np.quantile(means, 0.975)
verdict = "우세(CI>0)" if lo > 0 else "열세(CI<0)" if hi < 0 else "⚠️ 잡음과 구분 불가(CI가 0 포함)"
print(f"95% CI [{lo:+.4f}, {hi:+.4f}]  ->  {verdict}")

per-scenario Δ(agentic-facts): [0.19, 0.167, -0.5, 0.0, 0.0, -0.083, 0.833, 0.107]
평균 ΔMRR = +0.0893
95% CI [-0.1400, +0.3363]  ->  ⚠️ 잡음과 구분 불가(CI가 0 포함)


---
## 5. 1차 결론 (n=8 기준 — §9에서 후속 실측으로 갱신됨)

| 질문(§0) | 답 |
|---|---|
| ① 조립기 대신 LLM이 쓰면 검색 품질이 나아지는가 | **점추정은 최고(MRR .479)지만 n=8 CI가 0을 포함 — 확정 아님.** S07 한 시나리오(Δ+0.833)가 평균을 크게 끌어올렸다. |
| ② 정적 검증기가 과거 실제 버그를 잡아내는가 | **잡아낸다(7/7 단위 테스트 통과)** — 지금 production blob 쿼리 2건이 실제로 거절된다. |
| ③ 재시도/폴백 경로가 필요한 만큼 발동하는가 | **미확인** — 8/8이 1차 통과, 재시도 경로 자체가 이번 실행에서 한 번도 안 밟혔다. |

`.personal/strategy/RAG_05_에이전틱_쿼리_생성_설계.md` §3(미결정/한계)에 이 결과와 다음 단계
(의도적 실패 유발 실험, 실 IN_REVIEW 재검증)가 기록돼 있다. 이 시점의 노트북은 **프로토타입
실측**이지 프로덕션 반영 근거로 쓰기엔 아직 이르다 — 아래 §6~§9가 그 다음 단계를 실행한 결과다.

---
# 후속 실측 (2026-08-16, 근거 보완 — RAG_05 §3-A)

§5가 남긴 세 구멍(라벨 없는 raw 피처명 미노출·재시도 경로 미검증·표본 8건)을 추천 순서
(② raw 피처명 실노출 → ③ 의도적 실패 유발 → ① 표본 확대)대로 메운다.

## 6. 실험 2 — LLM이 raw 피처명을 실제로 볼 때도 가이드 규칙 2를 지키는가

§3의 실험은 사람이 미리 자연어로 풀어둔 `feature_hint_nl`만 LLM에 줬다 — 프로덕션 `_stage2()`가
실제로 만드는 형태(`model.feature_contribs()` 반환을 그대로 이어붙인 `"피처명(weight), ..."`
문자열)는 시험한 적이 없다. 그 원본 형태를 그대로 프롬프트에 넣어 재실행한다.

In [1]:
def synth_contribs_str(primary_feature: str) -> str:
    """실제 model.feature_contribs() 반환 형태(top3, weight 합=1)를 흉내낸 합성값."""
    fillers = [f for f in FEATURE_COLUMNS if f != primary_feature][:2]
    return f"{primary_feature}(0.52), {fillers[0]}(0.31), {fillers[1]}(0.17)"


def run_experiment2(max_retries: int = 1) -> pd.DataFrame:
    rows = []
    for sid, cat, merchant, feat_raw, feat_nl, facts_nl, rel in STAGE2_SCENARIOS:
        raw_contribs = synth_contribs_str(feat_raw)
        feedback = None
        query, ok, reason, attempts, fallback_used = "", False, None, 0, False
        first_attempt_ok = None
        for attempt in range(max_retries + 1):
            attempts = attempt + 1
            query = generate_query_agentic(cat, merchant, raw_contribs, facts_nl, feedback=feedback)
            ok, reason = validate_query(query)
            if attempt == 0:
                first_attempt_ok = ok
            if ok:
                break
            feedback = reason
        if not ok:
            query = build_query_facts_fallback(cat, merchant, feat_nl, facts_nl)
            fallback_used = True
        rows.append({"scenario_id": sid, "raw_contribs_shown": raw_contribs, "query": query,
                     "first_attempt_ok": first_attempt_ok, "final_ok": ok,
                     "attempts": attempts, "fallback_used": fallback_used})
    return pd.DataFrame(rows)


EXP2_DF = run_experiment2()
display(EXP2_DF[["scenario_id", "raw_contribs_shown", "first_attempt_ok", "fallback_used"]])
print(f"\n1차 시도 통과율: {EXP2_DF.first_attempt_ok.mean():.2f} ({EXP2_DF.first_attempt_ok.sum()}/{len(EXP2_DF)})")
print(f"폴백 발동률: {EXP2_DF.fallback_used.mean():.2f}")
print("\n--- 생성된 쿼리 ---")
for _, r in EXP2_DF.iterrows():
    print(f"[{r.scenario_id}] {r.query}")

1차 시도 통과율: 1.00 (8/8)
폴백 발동률: 0.00

--- 생성된 쿼리 ---
[S01] 강남모던바에서 회식비로 결제한 건인데 참석 인원이 2명입니다. 회식비로 인정되는 최소 인원 기준이 어떻게 되는지 확인하고 싶습니다.
[S02] 한정식당 청담점에서 청탁금지법 대상자와 함께 식사한 비용이 업무추진비로 적절한지 확인하고 싶습니다.
[S03] OO헤어살롱에서 발생한 출장비 결제에 대해 승인 시간대와 승인 금액이 적절한지 확인하고 싶습니다. 이와 관련된 규정을 찾아보고 싶습니다.
[S04] 고급레스토랑에서 발생한 업무추진비 결제에 대해 규정 위반 여부를 확인하고 싶습니다. 특히 카드 사용 금액과 승인 시간대에 대한 기준이 궁금합니다.
[S05] 일식오마카세에서 발생한 업무추진비 결제 건에 대해 최근 일주일 동안의 사용 빈도와 승인 시간대, 그리고 결제 금액이 적절한지 확인하고 싶습니다. 이와 관련된 규정을 찾아보고 싶습니다.
[S06] 명품매장에서 발생한 업무추진비 지출이 청탁금지법 대상자와 관련이 있습니다. 이 경우 선물로 지출할 수 있는 규정과 조건을 확인하고 싶습니다.
[S07] OO호텔에서 발생한 출장비 지출 건에 대해 규정에 따른 숙박비 지출 기준을 확인하고 싶습니다.
[S08] 이자카야에서 회식비로 결제한 건인데, 사전승인 여부를 확인할 수 있는 규정을 찾고 싶습니다.


**8/8 1차 통과, 폴백 0건.** `FEATURE_COLUMNS` 리터럴이 그대로 나타난 문장은 없다 — LLM이 전부
사람 말로 풀어썼다.

⚠️ **검증기 약점 하나가 드러났다**: S03·S04의 생성 문장이 `"승인 시간대"`·`"승인 금액"`처럼
**띄어쓰기가 들어간 표현**을 썼다. `FEATURE_TOKEN_RE`는 `승인시간대`(붙여쓰기) 리터럴만 정확히
매칭하므로, 이번엔 우연히 안 걸렸을 뿐 **띄어쓰기 하나로 검증기를 피해가는 경로가 이론상 열려
있다.** 자연스러운 한국어 표현이라 지금은 위험한 패턴이 아니지만, 검증기가 형태소 경계에 취약한
문자열 매칭이라는 사실은 RAG_05 §2.3의 한계로 남는다.

## 7. 실험 3 — 의도적 실패 유발: `category`에 코드 sentinel `GLOBAL`을 그대로 노출

RAG_04가 실측으로 잡았던 그 실패 조건(scope=`GLOBAL`이 렌더링에 리터럴로 새던 것, RAG_04 §1.2a)을
그대로 재현한다 — `category` 슬롯에 `"GLOBAL"`을 문자 그대로 넣고 생성시킨다.

In [1]:
ADVERSARIAL_SCENARIOS = [
    ("G01", "GLOBAL", "강남모던바", "최근 일주일간 결제 횟수가 급증한 점", "참석 인원 2명"),
    ("G02", "GLOBAL", "한정식당 청담점", "평소 대비 결제 금액이 크게 높은 점", "청탁금지법 대상자 참석, 지출유형 식사"),
    ("G03", "GLOBAL", "고급레스토랑", "카드 누적 사용액이 이례적으로 큰 점", ""),
    ("G04", "GLOBAL", "명품매장", "결제 금액이 이례적으로 큰 점", "지출유형 선물, 청탁금지법 대상자 참석"),
    ("G05", "GLOBAL", "OO호텔", "평소 대비 숙박 결제 금액이 높은 점", "지출유형 숙박"),
    ("G06", "GLOBAL", "이자카야", "해당 가맹점과의 첫 거래라는 점", "사전승인 미확인"),
]


def run_experiment3(max_retries: int = 1) -> pd.DataFrame:
    rows = []
    for sid, cat, merchant, feat_nl, facts_nl in ADVERSARIAL_SCENARIOS:
        feedback = None
        query, ok, reason, attempts, fallback_used = "", False, None, 0, False
        first_attempt_ok = None
        for attempt in range(max_retries + 1):
            attempts = attempt + 1
            query = generate_query_agentic(cat, merchant, feat_nl, facts_nl, feedback=feedback)
            ok, reason = validate_query(query)
            if attempt == 0:
                first_attempt_ok = ok
            if ok:
                break
            feedback = reason
        if not ok:
            query = build_query_facts_fallback(cat, merchant, feat_nl, facts_nl)
            fallback_used = True
        rows.append({"scenario_id": sid, "query": query, "first_attempt_ok": first_attempt_ok,
                     "fallback_used": fallback_used})
    return pd.DataFrame(rows)


EXP3_DF = run_experiment3()
display(EXP3_DF)
print(f"\n1차 시도 통과율: {EXP3_DF.first_attempt_ok.mean():.2f} ({EXP3_DF.first_attempt_ok.sum()}/{len(EXP3_DF)})")
print(f"폴백 발동률: {EXP3_DF.fallback_used.mean():.2f}")

1차 시도 통과율: 1.00 (6/6)
폴백 발동률: 0.00


**6/6 1차 통과, 폴백 0건.** `GLOBAL` 문자열이 그대로 나타난 생성 쿼리는 없다 — LLM이 `GLOBAL`을
"분류: GLOBAL"이라는 입력값으로만 받아들이고 검색 문장에는 맥락(가맹점·사유)만 반영했다.

### 7.1 종합 — 실험 2+3 = 14건의 의도적 스트레스 테스트, 위반 0건

정상 조건(§3의 원본 8건)까지 합치면 지금까지 실행된 28건 중 검증기가 실제로 거절한 사례는 없다.
**그런데 이게 곧 재시도/폴백 경로가 여전히 미검증이라는 뜻이기도 하다** — "가이드가 잘 지켜진다"는
확인됐지만, "안 지켜졌을 때 검증기·재시도·폴백이 실제로 작동하는지"는 이 노트북으로는 아직 답할
수 없다. 이번 실험은 "입력 재료에 위험한 값을 노출"하는 수준이었지 "LLM에게 위반을 직접
지시"하는 수준은 아니었다 — 후자가 다음으로 필요한 실험이다.

## 8. 실험 1 — 표본 확대 (n=8 → n=17)

`agent_gold_set.csv`(RAG_04 평가 노트북, 실제 커밋된 라벨)의 prohibition/limit_lookup 항목 중
기존 8개 시나리오와 인용 조문이 겹치지 않는 9건을 골라 Risk Review 프레이밍으로 재구성했다.
**가맹점명·이상탐지 사유는 이번에 새로 지었지만, 인용 조문(`relevant_units`)은 실제 커밋된 라벨을
그대로 썼다** — 새 규정 사실을 지어내지 않았다.

In [1]:
EXTRA_SCENARIOS = [
    ("X01", "법인카드", "OO상품권매장", "일시불할부구분코드",
     "일시불·할부 결제 패턴이 특이한 점", "법인카드로 상품권을 현금처럼 구매", ["법인카드_사용규정|제9조"]),
    ("X02", "법인카드", "부서 공용카드 결제", "사용자표준편차_확장",
     "결제 담당자가 자주 바뀌는 패턴을 보인 점", "부서 공용카드를 여러 부서원이 돌아가며 사용", ["법인카드_사용규정|제6조"]),
    ("X03", "업무추진비", "가나실업 접대", "통합승인금액",
     "결제 금액이 이례적으로 큰 점", "청탁금지법 한도를 초과한 접대로 추정됨", ["업무추진비_사용규정|제9조"]),
    ("X04", "업무추진비", "OO컨설팅 미팅", "카드누적사용액",
     "카드 누적 사용액이 큰 점", "건당 지출 금액이 커 증빙서류 기준 확인 필요", ["법인카드_사용규정|제11조"]),
    ("X05", "업무추진비", "한식당 회의", "시간대구간",
     "식사 시간대치고 이례적인 시간대 결제라는 점", "사전승인 없이 진행된 식사 자리로 추정됨", ["법인카드_사용규정|제10조"]),
    ("X06", "업무추진비", "OO식당", "월말여부",
     "지출 시점과 정산 등록 시점 사이 간격이 길어 보이는 점", "지출 후 정산 시스템 등록이 지연된 것으로 보임", ["법인카드_사용규정|제12조"]),
    ("X07", "법인카드", "본부장 개인 다수 결제", "최근7일사용횟수",
     "최근 결제 횟수가 급증한 점", "본부장 명의 카드의 결제 한도 초과 여부 확인 필요", ["법인카드_사용규정|별표1"]),
    ("X08", "출장비", "OO호텔 국내", "거래금액_Zscore_확장",
     "평소 대비 숙박 결제 금액이 높은 점", "국내 출장 1박 숙박", ["출장비_사용규정|별표1"]),
    ("X09", "출장비", "유럽출장 호텔", "카드첫거래여부",
     "해당 가맹점과의 첫 거래라는 점", "유럽 출장 중 숙박", ["출장비_사용규정|별표2"]),
]

ALL_SCENARIOS = STAGE2_SCENARIOS + EXTRA_SCENARIOS


def run_agentic_arm(scenarios, max_retries: int = 1) -> pd.DataFrame:
    rows = []
    for sid, cat, merchant, feat_raw, feat_nl, facts_nl, rel in scenarios:
        feedback = None
        query, ok, reason, attempts, fallback_used = "", False, None, 0, False
        for attempt in range(max_retries + 1):
            attempts = attempt + 1
            query = generate_query_agentic(cat, merchant, feat_nl, facts_nl, feedback=feedback)
            ok, reason = validate_query(query)
            if ok:
                break
            feedback = reason
        if not ok:
            query = build_query_facts_fallback(cat, merchant, feat_nl, facts_nl)
            fallback_used = True
        score = score_stage2(query, rel)
        score.update(scenario_id=sid, query=query)
        rows.append(score)
    return pd.DataFrame(rows)


def run_deterministic_arm(scenarios, builder) -> pd.DataFrame:
    rows = []
    for sid, cat, merchant, feat_raw, feat_nl, facts_nl, rel in scenarios:
        if builder is build_query_blob:
            q = builder(cat, merchant, feat_raw)
        elif builder is build_query_natural:
            q = builder(cat, merchant, feat_nl)
        else:
            q = builder(cat, merchant, feat_nl, facts_nl)
        score = score_stage2(q, rel)
        score.update(scenario_id=sid, query=q)
        rows.append(score)
    return pd.DataFrame(rows)


BLOB17 = run_deterministic_arm(ALL_SCENARIOS, build_query_blob)
NATURAL17 = run_deterministic_arm(ALL_SCENARIOS, build_query_natural)
FACTS17 = run_deterministic_arm(ALL_SCENARIOS, build_query_facts_fallback)
AGENTIC17 = run_agentic_arm(ALL_SCENARIOS)

SUMMARY17 = pd.DataFrame([
    {"방식": "블롭", "n": len(BLOB17), "MRR": BLOB17.mrr.mean(), "Recall@1": BLOB17["recall@1"].mean(), "nDCG@10": BLOB17["ndcg@10"].mean()},
    {"방식": "자연어", "n": len(NATURAL17), "MRR": NATURAL17.mrr.mean(), "Recall@1": NATURAL17["recall@1"].mean(), "nDCG@10": NATURAL17["ndcg@10"].mean()},
    {"방식": "자연어+facts", "n": len(FACTS17), "MRR": FACTS17.mrr.mean(), "Recall@1": FACTS17["recall@1"].mean(), "nDCG@10": FACTS17["ndcg@10"].mean()},
    {"방식": "에이전틱", "n": len(AGENTIC17), "MRR": AGENTIC17.mrr.mean(), "Recall@1": AGENTIC17["recall@1"].mean(), "nDCG@10": AGENTIC17["ndcg@10"].mean()},
]).round(4)
display(SUMMARY17)

       방식   n    MRR  Recall@1  nDCG@10
0      블롭  17 0.2520    0.1765   0.2911
1     자연어  17 0.2084    0.1176   0.2570
2 자연어+facts 17 0.3134    0.1765   0.3987
3     에이전틱  17 0.5163    0.4118   0.5685


⚠️ **n=8→17로 늘리자 "자연어가 블롭보다 낫다"는 §5의 결론이 뒤집혔다**(0.293→0.208로 자연어가
오히려 블롭 아래로 내려갔다). n=8이 방향성조차 얼마나 불안정한 표본이었는지 보여준다 — facts와
agentic 두 arm의 우위는 유지됐지만 blob/natural의 순위는 표본 크기에 민감했다.

### 8.1 에이전틱 vs 자연어+facts 짝비교(n=17) — 통계적으로 유의한가

In [1]:
delta17 = (AGENTIC17.sort_values("scenario_id").mrr.values
           - FACTS17.sort_values("scenario_id").mrr.values)
print("per-scenario Δ(agentic-facts):", np.round(delta17, 3).tolist())
print(f"평균 ΔMRR = {delta17.mean():+.4f}")

rng = np.random.default_rng(SEED)
idx = rng.integers(0, len(delta17), size=(5000, len(delta17)))
means = delta17[idx].mean(axis=1)
lo, hi = np.quantile(means, 0.025), np.quantile(means, 0.975)
verdict = "우세(CI>0)" if lo > 0 else "열세(CI<0)" if hi < 0 else "⚠️ 잡음과 구분 불가(CI가 0 포함)"
print(f"95% CI [{lo:+.4f}, {hi:+.4f}]  ->  {verdict}  (n={len(delta17)})")

per-scenario Δ(agentic-facts): [0.857, 0.167, 0.0, 0.0, 0.0, -0.083, 0.833, -0.014, 0.0, 1.0, 0.0, -0.143, 0.0, 0.0, 0.0, 0.667, 0.167]
평균 ΔMRR = +0.2030
95% CI [+0.0406, +0.3907]  ->  우세(CI>0)  (n=17)


**§5의 "확정 아님" 결론이 뒤집혔다.** n=8에서는 CI `[-0.140, +0.336]`로 0을 포함해 잡음과
구분 안 됐지만, n=17에서는 CI가 0을 완전히 벗어나 **통계적으로 유의한 우세**로 판정된다(RAG_04·
RAG_05가 채택 기준으로 쓰는 "95% CI가 0 미포함" 잣대를 그대로 적용).

---
## 9. 최종 결론 (§5 갱신)

| 질문 | §5(n=8) | §9(n=17 + 스트레스 테스트) |
|---|---|---|
| 검색 품질이 조립기보다 나은가 | 점추정만 우세, CI가 0 포함 | **점추정도 격차 확대(MRR .479→.516), CI가 0 미포함 — 통계적으로 유의** |
| 가이드 규칙 2(피처명 회피)가 지켜지는가 | 미시험(사람이 미리 풀어쓴 값만 줌) | **raw 피처명을 실제로 노출해도 8/8 지켜짐**(단, 띄어쓰기로 검증기를 피해갈 이론적 여지 발견) |
| 가이드 규칙 3(sentinel 회피)이 지켜지는가 | 미시험 | **`GLOBAL` 리터럴을 직접 노출해도 6/6 지켜짐** |
| 재시도/폴백 경로가 검증됐는가 | 미확인 | **여전히 미확인** — 28건 스트레스 테스트에서 검증기가 한 번도 안 거절함 |

**남은 유보**: 여전히 합성 시나리오(인용 조문만 실제 라벨, 나머지는 구성)이고, 실 IN_REVIEW
데이터 재검증은 안 됐다. 무엇보다 이 설계의 핵심 안전장치인 재시도→피드백→폴백 경로가 지금까지
단 한 번도 실행된 적이 없다 — "실패해도 안전하다"는 주장은 아직 증명되지 않았다. 다음 실험은
검증기 규칙을 우회하기 어렵게 LLM에게 **직접 위반을 지시**하는 적대적 프롬프트가 되어야 한다.

`.personal/strategy/RAG_05_에이전틱_쿼리_생성_설계.md` §3-A에 이 결과가 그대로 기록돼 있다.